In [6]:
# ============================================
# 1. Import Libraries
# ============================================

import pandas as pd
import numpy as np

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

In [7]:
# ============================================
# 2. Load Data
# ============================================

male_file = "male_file.xlsx"
female_file = "female_file.xlsx"

male = pd.read_excel(male_file)
female = pd.read_excel(female_file)

data = pd.concat([male, female], ignore_index=True)

In [8]:
# ============================================
# 3. Encode Sex
# ============================================

data["Sex"] = data["Sex"].map({"M":0, "F":1})

In [9]:
# ============================================
# 4. Feature Engineering (Derived Ratios)
# ============================================

data["Leaf_compactness"] = data["Largeur_limbe"] / data["longueur_limbe"]

data["Petiole_ratio"] = data["Longueur_petiole"] / data["Longueur_feuille"]

data["Leaflet_shape"] = data["Longueur_foliole"] / data["Largeur_foliole"]

data["Leaflet_density"] = data["Nombre_folioles"] / data["Longueur_feuille"]

data["Petiole_shape"] = data["largeur_petiole"] / data["Longueur_petiole"]

In [10]:
# ============================================
# 5. Prepare Dataset
# ============================================

target = "Sex"
group_column = "Tree_ID"

X = data.drop(columns=[target, group_column])
y = data[target]
groups = data[group_column]

In [11]:
# ============================================
# 6. Define Models
# ============================================

models = {

    "RandomForest": RandomForestClassifier(
        n_estimators=300,
        random_state=42
    ),

    "SVM": SVC(
        kernel="rbf",
        probability=True,
        random_state=42
    ),

    "LogisticRegression": LogisticRegression(
        max_iter=2000
    ),

    "GradientBoosting": GradientBoostingClassifier(
        random_state=42
    )
}

In [12]:
# ============================================
# 7. GroupKFold Cross Validation
# ============================================

gkf = GroupKFold(n_splits=5)

results = []

for model_name, model in models.items():

    acc_scores = []
    f1_scores = []
    precision_scores = []
    recall_scores = []

    for train_idx, test_idx in gkf.split(X, y, groups):

        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        pipeline = Pipeline([
            ("scaler", StandardScaler()),
            ("model", model)
        ])

        pipeline.fit(X_train, y_train)

        y_pred = pipeline.predict(X_test)

        acc_scores.append(accuracy_score(y_test, y_pred))
        f1_scores.append(f1_score(y_test, y_pred))
        precision_scores.append(precision_score(y_test, y_pred))
        recall_scores.append(recall_score(y_test, y_pred))


    results.append({

        "Model": model_name,
        "Accuracy": np.mean(acc_scores),
        "F1-score": np.mean(f1_scores),
        "Precision": np.mean(precision_scores),
        "Recall": np.mean(recall_scores)

    })

In [13]:
# ============================================
# 8. Display Results
# ============================================

results_df = pd.DataFrame(results)

print("\nModel Comparison:\n")
print(results_df.sort_values(by="Accuracy", ascending=False))


Model Comparison:

                Model  Accuracy  F1-score  Precision    Recall
1                 SVM  0.502980  0.391984   0.490858  0.349921
3    GradientBoosting  0.497878  0.442235   0.480437  0.416920
2  LogisticRegression  0.491267  0.416395   0.481109  0.387383
0        RandomForest  0.474569  0.431970   0.452911  0.425830
